# City of Cape Town Service Request Density Classification

This notebook analyzes City of Cape Town service request data to classify areas by population density using a 10-point decile scale. The assumption is that high volumes of service requests originate from densely-populated areas.

## Methodology
- **Data Sources**: Service request data (`sr_hex.csv`) joined with H3 hex polygons (`hex_polygons.geojson`)
- **Classification**: Areas are classified into deciles (1-10) based on service request volume
- **Output**: Interactive hex map with density coloring and filtering capabilities


## PART 1: Input Data Loading and Cleaning

In [1]:
import warnings
from datetime import datetime
from io import StringIO

import geopandas as gpd
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

### Load Service Request Data

Expected schema:
- `notification_number`: 12-digit string, not nullable
- `reference_number`: 10 digits (nullable)
- `creation_timestamp`, `completion_timestamp`: timestamps with +02:00 timezone
- `latitude`: float between -35 and -32
- `longitude`: float between 17 and 20
- `h3_level8_index`: 15 alphanumeric chars OR "0", not nullable

In [ ]:
DATA_PATH = 'raw_data/sr_hex.csv'
GEO_PATH = 'raw_data/hex_polygons.geojson'

EXPECTED_COLUMNS = [
    'notification_number',
    'reference_number',
    'creation_timestamp',
    'completion_timestamp',
    'directorate',
    'department',
    'branch',
    'section',
    'code_group',
    'code',
    'cause_code_group',
    'cause_code',
    'official_suburb',
    'latitude',
    'longitude',
    'h3_level8_index',
]

dtype_spec = {
    'notification_number': str,
    'reference_number': str,
    'directorate': str,
    'department': str,
    'branch': str,
    'section': str,
    'code_group': str,
    'code': str,
    'cause_code_group': str,
    'cause_code': str,
    'official_suburb': str,
    'h3_level8_index': str,
}

malformed_lines = []

def capture_bad_lines(bad_line):
    """Capture malformed lines for inspection. Returns None to skip the line."""
    malformed_lines.append(bad_line)
    return None

df_raw = pd.read_csv(
    DATA_PATH,
    dtype=dtype_spec,
    names=EXPECTED_COLUMNS,
    header=0,
    on_bad_lines=capture_bad_lines,
    engine='python'
)

print(f"Loaded {len(df_raw):,} records")
print(f"Captured {len(malformed_lines)} malformed lines")

Loaded 941,634 records
Captured 0 malformed lines


### Data Quality Report

In [ ]:
def generate_quality_report(df):
    """Generate comprehensive data quality report."""
    report = []
    
    report.append("=" * 60)
    report.append("DATA QUALITY REPORT")
    report.append("=" * 60)
    report.append(f"\nTotal Records: {len(df):,}")
    report.append(f"Total Columns: {len(df.columns)}")
    
    report.append("\n" + "-" * 60)
    report.append("NULL RATES PER FIELD")
    report.append("-" * 60)
    
    null_rates = df.isnull().sum() / len(df) * 100
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = null_rates[col]
        report.append(f"{col:30} | {null_count:>8,} nulls | {null_pct:>6.2f}%")
    
    report.append("\n" + "-" * 60)
    report.append("MIN/MAX VALUES FOR NUMERIC FIELDS")
    report.append("-" * 60)
    
    numeric_cols = ['latitude', 'longitude']
    for col in numeric_cols:
        if col in df.columns:
            min_val = df[col].min()
            max_val = df[col].max()
            report.append(f"{col:30} | min: {min_val:>15.6f} | max: {max_val:>15.6f}")
    
    return "\n".join(report)

print(generate_quality_report(df_raw))

DATA QUALITY REPORT

Total Records: 941,634
Total Columns: 16

------------------------------------------------------------
NULL RATES PER FIELD
------------------------------------------------------------
notification_number            |        0 nulls |   0.00%
reference_number               |  348,714 nulls |  37.03%
creation_timestamp             |        0 nulls |   0.00%
completion_timestamp           |   12,192 nulls |   1.29%
directorate                    |    9,435 nulls |   1.00%
department                     |    9,454 nulls |   1.00%
branch                         |   28,401 nulls |   3.02%
section                        |   93,125 nulls |   9.89%
code_group                     |        0 nulls |   0.00%
code                           |        0 nulls |   0.00%
cause_code_group               |  810,517 nulls |  86.08%
cause_code                     |  811,965 nulls |  86.23%
official_suburb                |  212,413 nulls |  22.56%
latitude                       |  212,36

### Data Validation Checks

In [ ]:
def validate_data(df):
    """Run validation checks and return issues found."""
    issues = {}
    
    invalid_notif = df[
        df['notification_number'].isnull() | 
        ~df['notification_number'].str.match(r'^\d{12}$', na=False)
    ]
    issues['invalid_notification_number'] = len(invalid_notif)
    
    original_creation = df['creation_timestamp'].copy()
    
    df['creation_timestamp'] = pd.to_datetime(
        df['creation_timestamp'], 
        errors='coerce',
        utc=True
    )

    
    unparseable_creation = df['creation_timestamp'].isna() & original_creation.notna()
    issues['unparseable_creation_timestamp'] = unparseable_creation.sum()
    
    cutoff_date = pd.Timestamp('2020-01-01', tz='Africa/Johannesburg')
    future_date = pd.Timestamp.now(tz='Africa/Johannesburg')
    
    invalid_creation_dates = df[
        df['creation_timestamp'].notna() & (
            (df['creation_timestamp'] < cutoff_date) | 
            (df['creation_timestamp'] > future_date)
        )
    ]
    issues['creation_timestamp_out_of_range'] = len(invalid_creation_dates)
    
    oob_lat = df[
        (df['latitude'] < -35) | (df['latitude'] > -32)
    ]
    issues['out_of_bounds_latitude'] = len(oob_lat)

    null_lat = df[
        df['latitude'].isnull()
    ]
    issues['null_latitude'] = len(null_lat)
    
    oob_lon = df[
        (df['longitude'] < 17) | (df['longitude'] > 20) 
    ]
    issues['out_of_bounds_longitude'] = len(oob_lon)

    null_lon = df[
        df['longitude'].isnull()
    ]
    issues['null_longitude'] = len(null_lon)
    
    invalid_h3 = df[
        df['h3_level8_index'].isnull() |
        ~df['h3_level8_index'].str.match(r'^([0-9a-fA-F]{15}|0)$', na=False)
    ]
    issues['invalid_h3_index'] = len(invalid_h3)
    
    return issues, df

issues, df = validate_data(df_raw.copy())

print("=" * 60)
print("VALIDATION ISSUES FOUND")
print("=" * 60)
for issue_type, count in issues.items():
    pct = count / len(df_raw) * 100
    print(f"{issue_type:40} | {count:>8,} records | {pct:>6.2f}%")




VALIDATION ISSUES FOUND
invalid_notification_number              |        0 records |   0.00%
unparseable_creation_timestamp           |        0 records |   0.00%
creation_timestamp_out_of_range          |        1 records |   0.00%
out_of_bounds_latitude                   |        0 records |   0.00%
null_latitude                            |  212,364 records |  22.55%
out_of_bounds_longitude                  |        0 records |   0.00%
null_longitude                           |  212,364 records |  22.55%
invalid_h3_index                         |        0 records |   0.00%


### Sample Malformed Lines

In [ ]:
print("=" * 60)
print("MALFORMED LINES SAMPLE")
print("=" * 60)

if malformed_lines:
    print(f"\nTotal malformed lines: {len(malformed_lines)}")
    print("\nFirst 5 malformed lines:")
    for i, line in enumerate(malformed_lines[:5]):
        print(f"\n--- Line {i+1} ---")
        print(line)
else:
    print("\nNo malformed lines detected during parsing.")

MALFORMED LINES SAMPLE

No malformed lines detected during parsing.


In [ ]:
print("=" * 60)
print("SAMPLE RECORDS WITH VALIDATION ISSUES")
print("=" * 60)

# Sample records for each validation issue

samples_to_display = {}

# Out-of-bounds latitude OR longitude (grouped together)
oob_latlon = df_raw[
    ((df_raw['latitude'] < -35) | (df_raw['latitude'] > -32)) |
    ((df_raw['longitude'] < 17) | (df_raw['longitude'] > 20))
]
if len(oob_latlon) > 0:
    samples_to_display['Out-of-bounds latitude or longitude'] = oob_latlon

# Creation timestamps out of reasonable range
cutoff_date = pd.Timestamp('2020-01-01', tz='Africa/Johannesburg')
future_date = pd.Timestamp.now(tz='Africa/Johannesburg')

invalid_creation_dates = df[
        df['creation_timestamp'].notna() & (
            (df['creation_timestamp'] < cutoff_date) | 
            (df['creation_timestamp'] > future_date)
        )
    ]
if len(invalid_creation_dates) > 0:
    samples_to_display['Bad or impossible timestamps'] = invalid_creation_dates


# Null latitude OR longitude (grouped together)
null_latlon = df_raw[df_raw['latitude'].isnull() | df_raw['longitude'].isnull()]
if len(null_latlon) > 0:
    samples_to_display['Null latitude or longitude'] = null_latlon

# Invalid H3 index
invalid_h3 = df_raw[
    df_raw['h3_level8_index'].isnull() |
    ~df_raw['h3_level8_index'].astype(str).str.match(r'^([0-9a-fA-F]{15}|0)$', na=False)
]
if len(invalid_h3) > 0:
    samples_to_display['Invalid H3 index'] = invalid_h3

# Display up to 5 records for each issue
if samples_to_display:
    for issue, df_issue in samples_to_display.items():
        print(f"\n{issue}: {len(df_issue)} record(s)")
        print("Sample (first 5):")
        display(
            df_issue[[
                'notification_number', 'latitude', 'longitude', 'code_group',
                'code', 'branch', 'section',
                'official_suburb', 'h3_level8_index'
            ]].head()
        )
else:
    print("\nNo records with validation issues.")

#TO DO: merge the validation checks with the sample records print

SAMPLE RECORDS WITH VALIDATION ISSUES

Null latitude or longitude: 212364 record(s)
Sample (first 5):


,notification_number,latitude,longitude,code_group,code,branch,section,official_suburb,h3_level8_index
13742,000400525315,NaN,NaN,TD Customer complaint groups,"RequestNewRoadway painted, mounted signs",RIM Area Central,District: Blaauwberg,NaN,0
13743,000400527116,NaN,NaN,TD Customer complaint groups,"RequestNewRoadway painted, mounted signs",RIM Area North,District : Bellville,NaN,0
13744,000400528840,NaN,NaN,TD Customer complaint groups,"RequestNewRoadway painted, mounted signs",RIM Area Central,District: Blaauwberg,NaN,0
13745,000400530412,NaN,NaN,TD Customer complaint groups,"RequestNewRoadway painted, mounted signs",NaN,NaN,NaN,0
13746,000400530772,NaN,NaN,TD Customer complaint groups,Paint Markings Lines&Signs,NaN,NaN,NaN,0


---

## PART 2: Exploration of Data

### Output 1: Service Request Code Statistics

Table showing for each service request code:
- `code`: the service request code
- `highest_suburb`: suburb with most requests for that code
- `highest_requests`: count from that suburb
- `median_requests`: median requests per suburb for that code
- `total_requests`: total requests with that code

In [ ]:
code_suburb_counts = df.groupby(['code', 'official_suburb']).size().reset_index(name='count')

def get_code_stats(group):
    """Calculate statistics for each code."""
    idx_max = group['count'].idxmax()
    return pd.Series({
        'highest_suburb': group.loc[idx_max, 'official_suburb'],
        'highest_requests': group.loc[idx_max, 'count'],
        'median_requests': group['count'].median(),
        'total_requests': group['count'].sum()
    })

code_stats = code_suburb_counts.groupby('code').apply(
    get_code_stats, include_groups=False
).reset_index()

code_stats = code_stats.sort_values('total_requests', ascending=False)

print("SERVICE REQUEST CODE STATISTICS")
print("=" * 80)
display(code_stats)

### Output 2: Suburb Summary with Deciles

Table showing for each suburb:
- `official_suburb`: suburb name
- `total_requests`: sum of all service requests
- `decile`: which decile (1-10) the suburb falls into by total requests

In [ ]:
suburb_totals = df.groupby('official_suburb').size().reset_index(name='total_requests')

suburb_totals['decile'] = pd.qcut(
    suburb_totals['total_requests'], 
    q=10, 
    labels=range(1, 11),
    duplicates='drop'
)

suburb_totals = suburb_totals.sort_values('total_requests', ascending=False)

print("SUBURB SUMMARY WITH DECILES")
print("=" * 80)
print(f"Total suburbs: {len(suburb_totals)}")
print(f"\nDecile distribution:")
print(suburb_totals['decile'].value_counts().sort_index())
print("\n")
display(suburb_totals)

### Output 3: Bar Chart - Service Requests by Code

Number of service requests (y-axis) by code (x-axis), ordered from highest to lowest.

In [ ]:
code_counts = df['code'].value_counts().reset_index()
code_counts.columns = ['code', 'count']

fig = px.bar(
    code_counts,
    x='code',
    y='count',
    title='Service Requests by Code (Ordered High to Low)',
    labels={'code': 'Service Request Code', 'count': 'Number of Requests'},
    color='count',
    color_continuous_scale='Viridis'
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=600,
    showlegend=False
)

fig.show()

### Output 4: Bar Chart - Service Requests by Suburb

Number of service requests (y-axis) by suburb (x-axis), ordered from highest to lowest.

In [ ]:
suburb_counts = df['official_suburb'].value_counts().reset_index()
suburb_counts.columns = ['suburb', 'count']

fig = px.bar(
    suburb_counts,
    x='suburb',
    y='count',
    title='Service Requests by Suburb (Ordered High to Low)',
    labels={'suburb': 'Official Suburb', 'count': 'Number of Requests'},
    color='count',
    color_continuous_scale='Viridis'
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=600,
    showlegend=False,
    xaxis={'categoryorder': 'total descending'}
)

fig.show()

### Output 5: 3D Bar Chart - Service Requests by Suburb and Code

3D visualization showing service request counts by suburb (x-axis), code (z-axis), with count on y-axis.
Limited to top 15 suburbs and top 15 codes for clarity. Uses a colorblind-friendly palette (plasma: cool blues to hot reds).

In [ ]:
TOP_N_SUBURBS = 15
TOP_N_CODES = 15

top_suburbs = df['official_suburb'].value_counts().head(TOP_N_SUBURBS).index.tolist()
top_codes = df['code'].value_counts().head(TOP_N_CODES).index.tolist()

df_filtered = df[
    df['official_suburb'].isin(top_suburbs) & 
    df['code'].isin(top_codes)
]

pivot_data = df_filtered.groupby(['official_suburb', 'code']).size().reset_index(name='count')

pivot_matrix = pivot_data.pivot(
    index='official_suburb', 
    columns='code', 
    values='count'
).fillna(0)

suburb_order = df_filtered['official_suburb'].value_counts().index.tolist()
code_order = df_filtered['code'].value_counts().index.tolist()

pivot_matrix = pivot_matrix.reindex(index=suburb_order, columns=code_order)

z_data = pivot_matrix.values
x_labels = pivot_matrix.columns.tolist()
y_labels = pivot_matrix.index.tolist()

fig = go.Figure(data=[go.Surface(
    z=z_data,
    x=list(range(len(x_labels))),
    y=list(range(len(y_labels))),
    colorscale='Plasma',
    colorbar=dict(title='Request Count')
)])

fig.update_layout(
    title=f'Service Requests by Suburb and Code (Top {TOP_N_SUBURBS} x Top {TOP_N_CODES})',
    scene=dict(
        xaxis=dict(
            title='Code',
            ticktext=x_labels,
            tickvals=list(range(len(x_labels))),
            tickangle=-45
        ),
        yaxis=dict(
            title='Suburb',
            ticktext=y_labels,
            tickvals=list(range(len(y_labels)))
        ),
        zaxis=dict(title='Number of Requests'),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    height=800,
    width=1000
)

fig.show()

---

## PART 3: Classification and Presentation

### Hex Classification by Population Density Decile

The classification approach:
1. Aggregate service request counts per H3 hex
2. Assign each hex to a decile (1-10) based on its total SR count
3. Decile 1 = sparsest (lowest SR volume), Decile 10 = densest (highest SR volume)

In [ ]:
hex_gdf = gpd.read_file(GEO_PATH)
print(f"Loaded {len(hex_gdf)} hex polygons")
print(f"Columns: {hex_gdf.columns.tolist()}")

In [ ]:
df_valid = df[df['h3_level8_index'] != '0'].copy()

hex_sr_counts = df_valid.groupby('h3_level8_index').agg({
    'notification_number': 'count',
    'official_suburb': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'
}).reset_index()

hex_sr_counts.columns = ['h3_index', 'sr_count', 'primary_suburb']

hex_sr_counts['decile'] = pd.qcut(
    hex_sr_counts['sr_count'],
    q=10,
    labels=range(1, 11),
    duplicates='drop'
)

hex_sr_counts['decile'] = hex_sr_counts['decile'].astype(int)

print(f"Hexes with service requests: {len(hex_sr_counts)}")
print(f"\nDecile distribution:")
print(hex_sr_counts['decile'].value_counts().sort_index())
print(f"\nSR count statistics by decile:")
display(hex_sr_counts.groupby('decile')['sr_count'].describe())

In [ ]:
hex_merged = hex_gdf.merge(
    hex_sr_counts,
    left_on='index',
    right_on='h3_index',
    how='left'
)

hex_merged['sr_count'] = hex_merged['sr_count'].fillna(0).astype(int)
hex_merged['decile'] = hex_merged['decile'].fillna(0).astype(int)
hex_merged['primary_suburb'] = hex_merged['primary_suburb'].fillna('No Data')

print(f"Total hexes in map: {len(hex_merged)}")
print(f"Hexes with SR data: {len(hex_merged[hex_merged['sr_count'] > 0])}")
print(f"Hexes without SR data: {len(hex_merged[hex_merged['sr_count'] == 0])}")

### Interactive Hex Map

Features:
- Hex polygons colored by density decile (1-10)
- Hover tooltips showing suburb name and SR count
- Checkboxes to filter by service request code
- Color scale: cool colors (blue/purple) for sparse areas, hot colors (orange/red) for dense areas

In [ ]:
hex_code_counts = df_valid.groupby(['h3_level8_index', 'code']).size().reset_index(name='code_count')

hex_with_codes = hex_sr_counts.copy()
unique_codes = df_valid['code'].unique().tolist()

for code in unique_codes:
    code_data = hex_code_counts[hex_code_counts['code'] == code]
    code_col = f'code_{code.replace(" ", "_").replace("&", "and").replace("/", "_")}'
    hex_with_codes = hex_with_codes.merge(
        code_data[['h3_level8_index', 'code_count']].rename(
            columns={'h3_level8_index': 'h3_index', 'code_count': code_col}
        ),
        on='h3_index',
        how='left'
    )
    hex_with_codes[code_col] = hex_with_codes[code_col].fillna(0).astype(int)

print(f"Prepared hex data with {len(unique_codes)} code columns")
print(f"Sample codes: {unique_codes[:5]}")

In [ ]:
import folium
from folium.plugins import GroupedLayerControl
from branca.colormap import LinearColormap

cape_town_center = [-33.9249, 18.4241]

m = folium.Map(
    location=cape_town_center,
    zoom_start=10,
    tiles='cartodbpositron'
)

colormap = LinearColormap(
    colors=['#313695', '#4575b4', '#74add1', '#abd9e9', '#e0f3f8', 
            '#fee090', '#fdae61', '#f46d43', '#d73027', '#a50026'],
    vmin=0,
    vmax=10,
    caption='Population Density Decile (1=Sparse, 10=Dense)'
)

hex_map_data = hex_gdf.merge(
    hex_with_codes,
    left_on='index',
    right_on='h3_index',
    how='left'
)
hex_map_data['sr_count'] = hex_map_data['sr_count'].fillna(0).astype(int)
hex_map_data['decile'] = hex_map_data['decile'].fillna(0).astype(int)
hex_map_data['primary_suburb'] = hex_map_data['primary_suburb'].fillna('No Data')

print(f"Map data prepared: {len(hex_map_data)} hexes")

In [ ]:
all_hexes_layer = folium.FeatureGroup(name='All Service Requests')

for idx, row in hex_map_data.iterrows():
    if row['sr_count'] > 0:
        color = colormap(row['decile'])
        
        tooltip_text = f"""
        <b>Suburb:</b> {row['primary_suburb']}<br>
        <b>Total Requests:</b> {row['sr_count']:,}<br>
        <b>Density Decile:</b> {row['decile']}
        """
        
        folium.GeoJson(
            row['geometry'].__geo_interface__,
            style_function=lambda x, c=color: {
                'fillColor': c,
                'color': '#333333',
                'weight': 0.5,
                'fillOpacity': 0.7
            },
            tooltip=folium.Tooltip(tooltip_text)
        ).add_to(all_hexes_layer)

all_hexes_layer.add_to(m)
print("Added all hexes layer")

In [ ]:
top_codes_for_filter = df_valid['code'].value_counts().head(10).index.tolist()

code_layers = {}

for code in top_codes_for_filter:
    code_col = f'code_{code.replace(" ", "_").replace("&", "and").replace("/", "_")}'
    
    if code_col not in hex_map_data.columns:
        continue
    
    layer = folium.FeatureGroup(name=code, show=False)
    
    hex_with_code = hex_map_data[hex_map_data[code_col] > 0]
    
    code_max = hex_with_code[code_col].max() if len(hex_with_code) > 0 else 1
    
    for idx, row in hex_with_code.iterrows():
        code_count = row[code_col]
        intensity = code_count / code_max if code_max > 0 else 0
        color = colormap(intensity * 10)
        
        tooltip_text = f"""
        <b>Code:</b> {code}<br>
        <b>Suburb:</b> {row['primary_suburb']}<br>
        <b>Code Requests:</b> {code_count:,}<br>
        <b>Total Requests:</b> {row['sr_count']:,}
        """
        
        folium.GeoJson(
            row['geometry'].__geo_interface__,
            style_function=lambda x, c=color: {
                'fillColor': c,
                'color': '#333333',
                'weight': 0.5,
                'fillOpacity': 0.7
            },
            tooltip=folium.Tooltip(tooltip_text)
        ).add_to(layer)
    
    layer.add_to(m)
    code_layers[code] = layer

print(f"Added {len(code_layers)} code-specific layers")

In [ ]:
colormap.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save('cape_town_density_map.html')
print("Map saved to 'cape_town_density_map.html'")

m

### Classification Summary

The hexes have been classified into 10 deciles based on service request volume:
- **Decile 1** (Blue): Sparsely populated areas with very few service requests
- **Decile 10** (Red): Densely populated areas with high service request volumes

**Methodology Rationale**: The assumption that service request volume correlates with population density is reasonable because:
1. More residents = more potential requesters
2. Higher density areas have more infrastructure requiring maintenance
3. Urban areas with higher population density typically generate more municipal service needs

**Limitations**:
- Industrial areas may have high SR counts but low residential population
- Some areas may have low reporting rates despite high population
- Infrastructure age/condition also affects SR volume independent of population